# 05 RNN/LSTM 序列建模

## 本节目标

本节学习如何把序列数据转换成监督学习任务，并使用 LSTM 做下一步预测。

你将完成：

- 构造带噪声的正弦序列。
- 使用滑动窗口生成输入和目标。
- 理解 `[batch, seq_len, features]`。
- 训练一个 LSTM 回归模型。
- 观察 loss 曲线和预测曲线。

## 背景问题

序列数据和普通表格数据最大的区别是：顺序有意义。时间序列、文本、语音都具有这种特点。

本节关注的问题是：模型如何利用过去一段时间的信息预测下一个值？

## 核心概念

- **时间步**：序列中的一个位置。
- **滑动窗口**：把连续一段历史作为输入，把后一个值作为目标。
- **隐状态**：模型对历史信息的压缩表示。
- **LSTM**：通过门控机制更稳定地保留和更新序列信息。
- **batch_first=True**：输入使用 `[batch, seq_len, features]` 格式。

初学者最容易混淆的是 `seq_len` 和 `features`：本实验中，一个时间步只有一个数值特征。

## 最小代码示例：构造序列

这里使用带噪声的正弦曲线。它有明显周期趋势，又包含一些随机扰动，适合观察预测效果。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

steps = np.linspace(0, 18 * np.pi, 460)
series = np.sin(steps) + 0.08 * np.random.randn(len(steps))
series = series.astype(np.float32)

plt.figure(figsize=(8, 3))
plt.plot(series)
plt.title("Synthetic sequence")
plt.xlabel("time")
plt.ylabel("value")
plt.grid(alpha=0.3)
plt.show()

## 最小代码示例：滑动窗口

窗口长度决定模型能看到多长的历史。窗口太短可能信息不够，窗口太长会增加训练难度。

In [ ]:
def make_windows(values, window_size=20):
    xs, ys = [], []
    for i in range(len(values) - window_size):
        xs.append(values[i:i + window_size])
        ys.append(values[i + window_size])
    x = torch.tensor(np.array(xs)).unsqueeze(-1)
    y = torch.tensor(np.array(ys)).unsqueeze(-1)
    return x, y


window_size = 20
x_all, y_all = make_windows(series, window_size)

print("x_all shape:", x_all.shape)
print("y_all shape:", y_all.shape)
print("one input window shape:", x_all[0].shape)

## 数据划分

时间序列通常不建议随机打乱后划分训练集和测试集，因为这样可能让模型提前看到未来信息。这里使用前 80% 训练，后 20% 测试。

In [ ]:
split = int(len(x_all) * 0.8)
x_train, y_train = x_all[:split], y_all[:split]
x_test, y_test = x_all[split:], y_all[split:]

print("train:", x_train.shape, y_train.shape)
print("test:", x_test.shape, y_test.shape)

## 完整实验：定义 LSTM 模型

LSTM 会输出每个时间步的隐藏表示。对于“用整个窗口预测下一个值”的任务，我们取最后一个时间步的输出，再接一个线性层。

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        sequence_output, _ = self.lstm(x)
        last_step = sequence_output[:, -1, :]
        return self.output(last_step)


model = LSTMRegressor(hidden_dim=32)
print(model)

## 完整实验：训练模型

这个任务是连续值预测，因此使用 MSELoss。为了保持 Notebook 简洁，这里直接用全量训练数据；更大数据集可以换成 DataLoader。

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []

for epoch in range(100):
    model.train()
    pred = model(x_train)
    loss = criterion(pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

model.eval()
with torch.no_grad():
    test_pred = model(x_test)
    test_loss = criterion(test_pred, y_test).item()

print(f"final train loss={losses[-1]:.4f}")
print(f"test loss={test_loss:.4f}")

## 实验观察：loss 曲线

从运行结果可以看到，loss 通常会下降到较低水平。如果 loss 大幅震荡，可以先降低学习率。

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("LSTM training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(alpha=0.3)
plt.show()

## 实验观察：预测曲线

预测曲线不需要逐点完全贴合目标，尤其当序列中存在噪声时。更重要的是看趋势是否跟随目标序列。

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(y_test.numpy()[:140], label="target")
plt.plot(test_pred.detach().numpy()[:140], label="prediction")
plt.title("One-step prediction")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 额外观察：窗口长度的影响

窗口长度是序列建模中的重要超参数。你可以尝试把 `window_size` 改成 5、10、40，观察训练速度和预测曲线变化。

In [ ]:
with torch.no_grad():
    one_window = x_test[:1]
    one_prediction = model(one_window)

print("one window shape:", one_window.shape)
print("one prediction shape:", one_prediction.shape)
print("prediction value:", one_prediction.item())
print("target value:", y_test[0].item())

## 常见错误

1. **输入缺少 feature 维度**：LSTM 输入应是三维。  
2. **忘记 `batch_first=True` 的影响**：输入维度顺序要和模型设置一致。  
3. **窗口和目标错位**：目标应该是窗口后面的下一个值。  
4. **随机划分时间序列**：可能造成信息泄漏。  
5. **学习率过大**：序列模型 loss 可能震荡明显。

调试时优先检查 `x_train.shape`、`y_train.shape`、模型输出 shape。

## 概念问答

**Q：为什么 LSTM 适合序列数据？**  
A：它按时间步处理输入，并通过隐状态保留历史信息。

**Q：为什么只取最后一个时间步？**  
A：本任务用整个历史窗口预测下一个值，最后时间步输出可以看作窗口信息的汇总。

**Q：时间序列一定不能随机划分吗？**  
A：不是绝对不能，但预测未来的任务通常应按时间顺序划分，避免泄漏。

**Q：预测曲线滞后怎么办？**  
A：可以尝试调整窗口长度、隐藏维度、训练轮数或换成多步预测设计。

## 小结

序列建模的核心不是只会调用 LSTM，而是理解如何构造样本、如何管理 shape、如何观察预测曲线。掌握这些后，可以继续扩展到文本分类、时间序列预测和 Transformer。